In [34]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
import os

load_dotenv()

True

In [35]:
model = ChatGroq(model='openai/gpt-oss-120b', api_key=os.getenv('GROQ_API_KEY'))

In [36]:
# define the state
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    
    sr: float
    bpb: float
    boundary_percentage: float
    summary: str

In [37]:
# calculte Strike rate
def calculate_sr(state: BatsmanState) -> BatsmanState:
    
    # Strike Rate = (Runs Scored / Balls Faced​) × 100
    
    runs = state['runs']
    balls = state['balls']
    
    if balls == 0:
        sr = 0
        
    sr = (runs/balls) * 100
    
    return {'sr': sr}

In [38]:
# calculate balls per boundary
def calculate_bpb(state: BatsmanState) ->  BatsmanState:
    
    # Balls per Boundary = Balls Faced / Number of Boundaries​
    
    balls = state['balls']
    fours = state['fours']
    sixes = state['sixes']
    
    boundaries = fours + sixes
    
    if boundaries == 0:
        state['bpb'] = 0
    
    bpb = balls/boundaries
    
    return {'bpb': bpb}

In [39]:
def calculate_boundary_percentage(state: BatsmanState) -> BatsmanState:
    
    # Boundary Percentage=(Runs from Boundaries​ / Total Run) × 100
    
    runs = state['runs']
    fours = state['fours']
    sixes = state['sixes']
    
    runs_from_boundaries = (fours * 4) + (sixes * 6)
    
    if runs == 0:
        state['boundary_percentage'] = 0
        
    boundary_percentage = (runs_from_boundaries / runs) * 100
    
    return {'boundary_percentage': boundary_percentage} 

In [40]:
def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
    Strike Rate: {state['sr']}\n
    Balls Per Boundary: {state['bpb']}\n
    Boundary Percentage: {state['boundary_percentage']}
    """
    
    return {'summary': summary}

In [41]:
# define the graph
graph = StateGraph(BatsmanState)

# add nodes to the graph
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percentage', calculate_boundary_percentage)
graph.add_node('summary', summary)

# add edges to the graph
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percentage')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percentage', 'summary')

graph.add_edge('summary', END)

# compile the graph
workflow = graph.compile()

In [42]:
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

final_state = workflow.invoke(initial_state)

print(final_state)

{'runs': 100, 'balls': 50, 'fours': 6, 'sixes': 4, 'sr': 200.0, 'bpb': 5.0, 'boundary_percentage': 48.0, 'summary': '\n    Strike Rate: 200.0\n\n    Balls Per Boundary: 5.0\n\n    Boundary Percentage: 48.0\n    '}


In [43]:
print(final_state['summary'])


    Strike Rate: 200.0

    Balls Per Boundary: 5.0

    Boundary Percentage: 48.0
    
